In [2]:
from functions import *
import matplotlib.ticker as ticker
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.ticker import FormatStrFormatter
import matplotlib as mpl
from matplotlib import cm
import h5py
import numpy as np
plt.close('all')

%matplotlib widget

FILE="output/chi2_SHEN_6param_eL_adjreso_z0.5.h5py"
f = h5py.File(FILE, "r") 
siglnX2 = f['siglnX2'][:]
siglnX1 = f['siglnX1'][:]
chi2_grid = f['chi2_grid'][:].T
f.close()

for index, value in np.ndenumerate(chi2_grid):
    if siglnX2[index[3]] > siglnX1[index[4]]:
        chi2_grid[index] = 1e10
        
null = np.where(chi2_grid == 1e10)

In [64]:
def Shen_fit_uncer(z, lums): ###best fit data from Shen+2020

    def get_params(params):
        rand_params = np.zeros((NUM, len(params)))
        ind = 0
        for p in params:
            i = np.random.randint(1,3,NUM)
            rand_params[:,ind][i == 1] = param_list[ind][0] + np.abs(np.random.normal(0, param_list[ind][1], size = len(i[i==1])))
            rand_params[:,ind][i == 2] = param_list[ind][0] - np.abs(np.random.normal(0, param_list[ind][2], size = len(i[i==2])))
            ind += 1
        return rand_params
    
    def shen_func_a(p):
        L = lums
        a0, a1, a2, b0, b1, b2, c0, c1, c2, d0, d1 = p
        zr = 2.0
        zfrac = (1 + z)/(1 + zr)
        g1 = C.chebval(1 + z, [a0, a1, a2])
        g2 = 2*b0/(zfrac**b1 + zfrac **b2)
        logLs = 2*c0/(zfrac**c1 + zfrac**c2)
        logPhis = C.chebval(1 + z, [d0, d1])
        Lfrac = 10**L / 10**logLs
        Phibol = 10**logPhis/(Lfrac**g1 + Lfrac**g2)

        return np.log10(Phibol)
    
    params_a = {'a0':[0.8569, 0.0247, 0.0253], 'a1':[-0.2614, 0.0162, 0.0164], 'a2':[0.0200,0.0011,0.0011],\
        'b0':[2.5375, 0.0177, 0.0187], 'b1':[-1.0425,0.0164, 0.0182], 'b2':[1.1201, 0.0199, 0.0207],\
        'c0':[13.0088, 0.0090, 0.0091], 'c1':[-0.5759, 0.0018, 0.0020], 'c2':[0.4554, 0.0028, 0.0027],\
        'd0':[-3.5426, 0.0235, 0.0209], 'd1':[-0.3936, 0.0070, 0.0073]}
    

    param_list = np.array([params_a[i] for i in params_a])
    NUM = int(1e4)
    rand_params = get_params(params_a)
    ys = np.apply_along_axis(shen_func_a, 1, rand_params).T
    ya = shen_func_a(param_list[:,0]) 


    fracs = sp.stats.norm.cdf([-2, -1, 0, 1, 2])
    percs = np.percentile(ys, 100*fracs, axis=1)

    std_ave = np.std(ys, axis=1)
    std_blw = ya-percs[1,:]
    std_abv = percs[3,:]-ya

    return ya, std_ave, std_abv, std_blw





def QLFwShen_test(ax, z, qlf_params):
    
    qlf = QLF(z, qlf_params['bins'])
    qlf.get_Mbh( qlf_params['Mscrit'], qlf_params['preslope'], qlf_params['Mbhnorm'], approx_local=True, norm_local = 11+qlf_params['postnorm'])
    qlf.get_dNdlnL(qlf_params['lums'], [qlf_params['presig'], qlf_params['postsig']])
    
    lumsp = 10**qlf_params['lums']*3.8e33
    prea = np.zeros(len(lumsp))
    posta = np.zeros(len(lumsp))

    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF')
    
    pars = 6
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.5, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -10, np.log10(temp * np.log(10)), color = c, alpha = 0.2)
        mass_begin += 1

        
    xshen = 10**qlf_params['lums']*3.8e33
    dens, stanave, stanab, stanb = Shen_fit_uncer(z, qlf_params['lums'])
    
    ax.plot(xshen, dens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
    ax.fill_between(xshen, dens-stanab-0.5, dens+stanb+0.5, color='purple', alpha=.2)
    
    ax.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**48.75,-5,'z = '+str(z),fontsize = 12)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    ax.legend(fontsize = 8)
    
    
    
def P(chi2, N, p): #works only for small nu
    v = N - p
    return chi2**((v-2)/2)*np.exp(-chi2/2)

def get_AIC(chi2, N, p):
    return chi2 + 2*p + 2*p*(p+1)/(N-p-1)

# 2 Param Explorations

In [65]:
def bestfit_QLF_2param(z, qlf_params, pdf):
    
    pfs = 8
    pars = 6
    cs = list(cm.Greens(np.arange(pars) / pars) ) 

    fig, (ax, ax4) = plt.subplots(1,2,figsize=(8,3.5))

    
    qlf = QLF(z, qlf_params['bins'])
    qlf.get_Mbh( qlf_params['Mscrit'], qlf_params['preslope'], qlf_params['Mbhnorm'], approx_local=True, norm_local = 11+qlf_params['postnorm'])
    qlf.get_dNdlnL(qlf_params['lums'], [qlf_params['presig'], qlf_params['postsig']])
    
    shendens, stanave, stanab, stanb = Shen_fit_uncer(z, qlf_params['lums'])
    
    lumsp = 10**qlf_params['lums']*3.8e33
    
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.5, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -10, np.log10(temp * np.log(10)), color = c, alpha = 0.2)
        mass_begin += 1
        
    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF') 
    ax.plot(lumsp, shendens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
    ax.fill_between(lumsp, shendens-stanab-0.5, shendens+stanb+0.5, color='purple', alpha=.2)
    
    ax.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**48.75,-1,'z = '+str(z),fontsize = pfs)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    ax.legend(fontsize = 8)
    
    ax.axvline(10**7.5*3.8e33,c='r',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')
    
    minchi2 = qlf_params['minchi2']
    AIC = qlf_params['AIC']
    ax.text(10**48.75,-3, r'$\chi^2$ = '+f'{minchi2:.2f}', fontsize = pfs)
    if AIC:
        ax.text(10**48.75, -4, r'AIC = '+f'{AIC:.2f}', fontsize = pfs)

    
    ax4.plot(10**qlf.StellBins, 10**qlf.BHBins, c = 'k', label='Implemented')
    ax4.plot(10**qlf.StellBins, 10**(qlf.StellBins - 2.8), c = 'r', linestyle='dashed', label='Observed')
    for c, mass in zip(cs, [7,8,9,10,11,12,13]):
        ax4.axvline(10**mass, ls = 'dotted', c=c)
        ax4.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)
    
    bestpost, bestlocal = qlf_params['postsig'], qlf_params['postnorm']
    ax4.text(10**7.2, 10**7.7, '$\sigma_{ln X} = $'+f'{bestpost:.2f}', fontsize = pfs)
    ax4.text(10**8, 10**4.2, r'post-disk normalization = '+f'{10**bestlocal:.4f}', fontsize = pfs)
    
    ax4.set_xlabel('$M_*(M_\odot)$',fontsize=12)
    ax4.set_ylabel('$M_{BH}(M_\odot)$',fontsize=12)
    ax4.set_yscale('log')
    ax4.set_xscale('log')
    ax4.set_xlim([10**7,10**12.5])
    ax4.set_ylim([10**3, 10**10])
    ax4.legend(fontsize = 8, framealpha=1)
    ax4.tick_params(axis='both', which='both', labelsize=10, direction='in')

    AIC_tot = qlf_params['AIC_tot']
    chi2_tot = qlf_params['chi2_tot']
    plt.suptitle('2 Param SHEN Best Fit (force linear Mstar-Mbh)\n'+r'AIC$_{\rmtot}$ = '+f'{AIC_tot:.2f}'+r'    $\chi^2_{\rmtot} = $'+f'{chi2_tot:.2f}')
    plt.tight_layout(rect=[0, 0.03, 1, 0.9])

    pdf.savefig(fig)


## all free with z fits

In [66]:
plt.close('all')
%matplotlib widget
redshifts = ['0.5', '1.0', '2.0', '3.0', '4.0']


minval_tot = 0
for z in redshifts:
    FILE="output_072720/chi2_SHEN_2param_eL_medres_z"+z+".h5py"
    f = h5py.File(FILE, "r") 
    chi2_grid = f['chi2_grid'][:].T
    f.close()

    minval_tot += np.amin(chi2_grid)
AIC_tot = get_AIC(minval_tot, 100*len(redshifts), 2*len(redshifts))

with PdfPages('plots/bestfit_plots/extendof_kinkpref/QLF_forcelinear_freeall.pdf') as pdf:
    for z in redshifts:
        FILE="output_072720/chi2_SHEN_2param_eL_medres_z"+z+".h5py"
        f = h5py.File(FILE, "r") 
        logMstar0 = f['logMstar0'][:]
        siglnX2 = f['siglnX2'][:]
        siglnX1 = f['siglnX1'][:]
        slope_low = f['slope_low'][:]
        norm_from_local = f['norm_from_local'][:]
        norm_of_local = f['norm_of_local'][:]
        chi2_grid = f['chi2_grid'][:].T
        f.close()

        minval = np.amin(chi2_grid)
        minind = np.where(chi2_grid == minval)
        AIC = get_AIC(minval, 100, 2)

        bestlocal = norm_of_local[minind[0][0]] 
        bestnorm = norm_from_local[0]
        bestslope = slope_low[0]
        bestpost = siglnX2[minind[1][0]]
        bestpre = siglnX1[0]
        bestcrit = logMstar0[0]

        print(f'\nShen best fits (z = {z}): AIC =',AIC, r'chi2 =',minval)
        print(f'Best post-disk sigma = {bestpost:.2f}, Best local norm = {bestlocal:.2f}\n ')

        qlf_params = {'bins':0.005, 'Mscrit':bestcrit, 'Mbhnorm':bestnorm, 'preslope':bestslope, 'presig':bestpre, 'postsig':bestpost, 'postnorm':bestlocal, 'lums':np.linspace(5,18,200),\
                      'minchi2':minval, 'chi2_tot':minval_tot, 'AIC':AIC, 'AIC_tot':AIC_tot}
        
        bestfit_QLF_2param(float(z), qlf_params, pdf)



Shen best fits (z = 0.5): AIC = 9.337988309503936 chi2 = 5.214276969297751
Best post-disk sigma = 2.42, Best local norm = -3.00
 


FigureCanvasNbAgg()


Shen best fits (z = 1.0): AIC = 14.938556129183699 chi2 = 10.814844788977513
Best post-disk sigma = 2.26, Best local norm = -3.00
 


FigureCanvasNbAgg()


Shen best fits (z = 2.0): AIC = 35.93362993748326 chi2 = 31.809918597277072
Best post-disk sigma = 2.11, Best local norm = -3.00
 


FigureCanvasNbAgg()


Shen best fits (z = 3.0): AIC = 46.72905560278501 chi2 = 42.605344262578825
Best post-disk sigma = 2.42, Best local norm = -3.00
 


FigureCanvasNbAgg()


Shen best fits (z = 4.0): AIC = 41.305378348889484 chi2 = 37.1816670086833
Best post-disk sigma = 2.42, Best local norm = -3.00
 


FigureCanvasNbAgg()

## none free with z fits

In [68]:
plt.close('all')
%matplotlib widget

redshifts = ['0.5', '1.0', '2.0', '3.0', '4.0']
for z in redshifts:
    FILE="output_072720/chi2_SHEN_2param_eL_medres_z"+z+".h5py"
    f = h5py.File(FILE, "r") 
    logMstar0 = f['logMstar0'][:]
    siglnX2 = f['siglnX2'][:]
    siglnX1 = f['siglnX1'][:]
    slope_low = f['slope_low'][:]
    norm_from_local = f['norm_from_local'][:]
    norm_of_local = f['norm_of_local'][:]
    if z == redshifts[0]:
        chi2_grid = f['chi2_grid'][:].T
    else:
        chi2_grid += f['chi2_grid'][:].T
    f.close()
    
minval_tot = np.amin(chi2_grid)
minind = np.where(chi2_grid == minval_tot)
AIC_tot = get_AIC(minval_tot, 100*len(redshifts), 2)

bestlocal = norm_of_local[minind[0][0]] 
bestnorm = norm_from_local[0]
bestslope = slope_low[0]
bestpost = siglnX2[minind[1][0]]
bestpre = siglnX1[0]
bestcrit = logMstar0[0]

print(f'\nShen best fits (z = {z}):',r'AIC tot =',AIC_tot, r'chi2 tot =',minval_tot)
print(f'Best post-disk sigma = {bestpost:.2f}, Best local norm = {bestlocal:.2f}\n ')

qlf_params = {'bins':0.005, 'Mscrit':bestcrit, 'Mbhnorm':bestnorm, 'preslope':bestslope, 'presig':bestpre, 'postsig':bestpost, 'postnorm':bestlocal, 'lums':np.linspace(5,18,200),\
              'minchi2':None, 'chi2_tot':minval_tot, 'AIC':None, 'AIC_tot':AIC_tot}
    
    

with PdfPages('plots/bestfit_plots/extendof_kinkpref/QLF_forcelinear_freenone.pdf') as pdf:
    for z in redshifts:
        FILE="output_072720/chi2_SHEN_2param_eL_medres_z"+z+".h5py"
        f = h5py.File(FILE, "r") 
        chi2_grid = f['chi2_grid'][:].T
        f.close()

        minval = chi2_grid[minind][0]
        AIC = get_AIC(minval, 100, 2)

        qlf_params['minchi2'] = minval
        qlf_params['AIC'] = AIC
        
        bestfit_QLF_2param(float(z), qlf_params, pdf)



Shen best fits (z = 4.0): AIC tot = 141.14689154431724 chi2 tot = 137.12274667510195
Best post-disk sigma = 2.26, Best local norm = -3.00
 


FigureCanvasNbAgg()

FigureCanvasNbAgg()

FigureCanvasNbAgg()

FigureCanvasNbAgg()

FigureCanvasNbAgg()

# 6 Param Explorations

In [61]:
def bestfit_QLF_6param(z, qlf_params, pdf):
    
    pfs = 8
    pars = 6
    cs = list(cm.Greens(np.arange(pars) / pars) ) 

    fig, (ax, ax4) = plt.subplots(1,2,figsize=(8,3.5))

    
    qlf = QLF(z, qlf_params['bins'])
    qlf.get_Mbh( qlf_params['Mscrit'], qlf_params['preslope'], qlf_params['Mbhnorm'], approx_local=True, norm_local = 11+qlf_params['postnorm'])
    qlf.get_dNdlnL(qlf_params['lums'], [qlf_params['presig'], qlf_params['postsig']])
    
    shendens, stanave, stanab, stanb = Shen_fit_uncer(z, qlf_params['lums'])
    
    lumsp = 10**qlf_params['lums']*3.8e33
    
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.5, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -10, np.log10(temp * np.log(10)), color = c, alpha = 0.2)
        mass_begin += 1
        
    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF') 
    ax.plot(lumsp, shendens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
    ax.fill_between(lumsp, shendens-stanab-0.5, shendens+stanb+0.5, color='purple', alpha=.2)
    
    ax.axis([10**6*3.8e33,10**18*3.8e33,-10,0])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**48.75,-1,'z = '+str(z),fontsize = pfs)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    ax.legend(fontsize = 8)
    
    ax.axvline(10**7.5*3.8e33,c='r',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')
    
    minchi2 = qlf_params['minchi2']
    AIC = qlf_params['AIC']
    ax.text(10**48.75,-3, r'$\chi^2$ = '+f'{minchi2:.2f}', fontsize = pfs)
    if AIC:
        ax.text(10**48.75, -4, r'AIC = '+f'{AIC:.2f}', fontsize = pfs)

    
    ax4.plot(10**qlf.StellBins, 10**qlf.BHBins, c = 'k', label='Implemented')
    ax4.plot(10**qlf.StellBins, 10**(qlf.StellBins - 2.8), c = 'r', linestyle='dashed', label='Observed')
    for c, mass in zip(cs, [7,8,9,10,11,12,13]):
        ax4.axvline(10**mass, ls = 'dotted', c=c)
        ax4.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)
    
    
    bestcrit, bestpost, bestpre, bestslope, bestnorm, bestlocal = qlf_params['Mscrit'], qlf_params['postsig'], qlf_params['presig'], qlf_params['preslope'], qlf_params['Mbhnorm'], qlf_params['postnorm']
    ax4.text(10**7.2, 10**8.2, 'transition point = $10^{'+f'{bestcrit:.2f}'+'} M_\odot$', fontsize = pfs)
    ax4.text(10**7.2, 10**7.7, 'pre-disk $\sigma_{ln X} = $'+f'{bestpre:.2f}', fontsize = pfs)
    ax4.text(10**7.2, 10**7.2, 'post-disk $\sigma_{ln X} = $'+f'{bestpost:.2f}', fontsize = pfs)
    ax4.text(10**10.0, 10**5.2, r'pre-disk slope = '+f'{bestslope:.3f}', fontsize = pfs)
    ax4.text(10**8.5, 10**4.7, r'pre-disk normalization = $10^{'+f'{bestnorm:.2f}'+'} M_\odot$', fontsize = pfs)
    ax4.text(10**8.5, 10**4.2, r'post-disk normalization = '+f'{10**bestlocal:.4f}', fontsize = pfs)

    
    ax4.set_xlabel('$M_*(M_\odot)$',fontsize=12)
    ax4.set_ylabel('$M_{BH}(M_\odot)$',fontsize=12)
    ax4.set_yscale('log')
    ax4.set_xscale('log')
    ax4.set_xlim([10**7,10**12.5])
    ax4.set_ylim([10**3, 10**10])
    ax4.legend(fontsize = 8, framealpha=1)
    ax4.tick_params(axis='both', which='both', labelsize=10, direction='in')
    
    AIC_tot = qlf_params['AIC_tot']
    chi2_tot = qlf_params['chi2_tot']
    plt.suptitle('6 Param SHEN Best Fit (allow kink Mstar-Mbh)\n'+r'AIC$_{\rmtot}$ = '+f'{AIC_tot:.2f}'+r'    $\chi^2_{\rmtot} = $'+f'{chi2_tot:.2f}')
    plt.tight_layout(rect=[0, 0.03, 1, 0.9])

    pdf.savefig(fig)


## all free with z fits

In [62]:
plt.close('all')
%matplotlib widget
redshifts = ['0.5', '1.0', '2.0', '3.0', '4.0']

minval_tot = 0
for z in redshifts:
    FILE="output/chi2_SHEN_6param_eL_adjreso_z"+z+".h5py"
    f = h5py.File(FILE, "r") 
    chi2_grid = f['chi2_grid'][:].T
    f.close()
    
    chi2_grid[null] = 1e10
    minval_tot += np.amin(chi2_grid)
    
AIC_tot = get_AIC(minval_tot, 100*len(redshifts), 6*len(redshifts))


with PdfPages('plots/bestfit_plots/extendof_kinkpref/QLF_allowkink_freeall.pdf') as pdf:
    for z in redshifts:
        FILE="output/chi2_SHEN_6param_eL_adjreso_z"+z+".h5py"
        f = h5py.File(FILE, "r") 
        logMstar0 = f['logMstar0'][:]
        siglnX2 = f['siglnX2'][:]
        siglnX1 = f['siglnX1'][:]
        slope_low = f['slope_low'][:]
        norm_from_local = f['norm_from_local'][:]
        norm_of_local = f['norm_of_local'][:]
        chi2_grid = f['chi2_grid'][:].T
        f.close()
        
        chi2_grid[null] = 1e10

        minval = np.amin(chi2_grid)
        minind = np.where(chi2_grid == minval)
        AIC = get_AIC(minval, 100, 6)

        bestlocal = norm_of_local[minind[0][0]] 
        bestnorm = norm_from_local[minind[1][0]]
        bestslope = slope_low[minind[2][0]]
        bestpost = siglnX2[minind[3][0]]
        bestpre = siglnX1[minind[4][0]]
        bestcrit = logMstar0[minind[5][0]]
    
        print(f'\nShen best fits (z = {z}): AIC =',AIC, r'chi2 =',minval)
        print(f'Best crit = {bestcrit:.2f},  Best pre-disk = {bestpre:.2f}, Best post-disk = {bestpost:.2f},  Best slope = {bestslope:.2f},  Best norm = {bestnorm:.2f},   Best local norm = {bestlocal:.2f}\n ')

        qlf_params = {'bins':0.005, 'Mscrit':bestcrit, 'Mbhnorm':bestnorm, 'preslope':bestslope, 'presig':bestpre, 'postsig':bestpost, 'postnorm':bestlocal, 'lums':np.linspace(5,18,200),\
                      'minchi2':minval, 'chi2_tot':minval_tot, 'AIC':AIC, 'AIC_tot':AIC_tot}
        
        bestfit_QLF_6param(float(z), qlf_params, pdf)



Shen best fits (z = 0.5): AIC = 13.912263995797511 chi2 = 1.0090381893459
Best crit = 10.71,  Best pre-disk = 2.29, Best post-disk = 1.43,  Best slope = 0.82,  Best norm = 1.75,   Best local norm = -2.30
 


FigureCanvasNbAgg()


Shen best fits (z = 1.0): AIC = 14.099931275727002 chi2 = 1.1967054692753898
Best crit = 10.71,  Best pre-disk = 1.86, Best post-disk = 1.86,  Best slope = 0.96,  Best norm = 1.93,   Best local norm = -2.35
 


FigureCanvasNbAgg()


Shen best fits (z = 2.0): AIC = 14.507869456312847 chi2 = 1.6046436498612344
Best crit = 10.71,  Best pre-disk = 1.64, Best post-disk = 1.64,  Best slope = 1.30,  Best norm = 2.11,   Best local norm = -2.40
 


FigureCanvasNbAgg()


Shen best fits (z = 3.0): AIC = 14.746204616213944 chi2 = 1.8429788097623319
Best crit = 10.71,  Best pre-disk = 1.86, Best post-disk = 1.43,  Best slope = 1.30,  Best norm = 1.93,   Best local norm = -2.35
 


FigureCanvasNbAgg()


Shen best fits (z = 4.0): AIC = 13.64597440491512 chi2 = 0.7427485984635087
Best crit = 10.34,  Best pre-disk = 2.29, Best post-disk = 1.64,  Best slope = 1.30,  Best norm = 1.57,   Best local norm = -2.60
 


FigureCanvasNbAgg()

## none free with z fits

In [54]:
plt.close('all')
%matplotlib widget

redshifts = ['0.5', '1.0', '2.0', '3.0', '4.0']
for z in redshifts:
    FILE="output-2P/chi2_SHEN_6param_eL_adjreso_z"+z+".h5py"
    f = h5py.File(FILE, "r") 
    logMstar0 = f['logMstar0'][:]
    siglnX2 = f['siglnX2'][:]
    siglnX1 = f['siglnX1'][:]
    slope_low = f['slope_low'][:]
    norm_from_local = f['norm_from_local'][:]
    norm_of_local = f['norm_of_local'][:]
    if z == redshifts[0]:
        chi2_grid = f['chi2_grid'][:].T
    else:
        chi2_grid += f['chi2_grid'][:].T
    f.close()

print(logMstar0)

chi2_grid[null] = 1e10
    
minval_tot = np.amin(chi2_grid)
minind = np.where(chi2_grid == minval_tot)
AIC_tot = get_AIC(minval_tot, 100*len(redshifts), 6)

bestlocal = norm_of_local[minind[0][0]] 
bestnorm = norm_from_local[minind[1][0]]
bestslope = slope_low[minind[2][0]]
bestpost = siglnX2[minind[3][0]]
bestpre = siglnX1[minind[4][0]]
bestcrit = logMstar0[minind[5][0]]

print(f'\nShen best fits (z = {z}): AIC =',AIC, r'chi2 =',minval)
print(f'Best crit = {bestcrit:.2f},  Best pre-disk = {bestpre:.2f}, Best post-disk = {bestpost:.2f},  Best slope = {bestslope:.2f},  Best norm = {bestnorm:.2f},   Best local norm = {bestlocal:.2f}\n ')

qlf_params = {'bins':0.005, 'Mscrit':bestcrit, 'Mbhnorm':bestnorm, 'preslope':bestslope, 'presig':bestpre, 'postsig':bestpost, 'postnorm':bestlocal, 'lums':np.linspace(5,18,200),\
              'minchi2':minval, 'AIC':None, 'chi2_tot':minval_tot, 'AIC_tot':AIC_tot}
      

with PdfPages('plots/bestfit_plots/extendof_kinkpref/QLF_allowkink_freenone.pdf') as pdf:
    for z in redshifts:
        FILE="output/chi2_SHEN_6param_eL_adjreso_z"+z+".h5py"
        f = h5py.File(FILE, "r") 
        chi2_grid = f['chi2_grid'][:].T
        f.close()

        minval = chi2_grid[minind][0]
        AIC = get_AIC(minval, 100, 6)
        
        qlf_params['minchi2'] = minval
        qlf_params['AIC'] = AIC
        
#         bestfit_QLF_6param(float(z), qlf_params, pdf)


[ 8.5         8.68421053  8.86842105  9.05263158  9.23684211  9.42105263
  9.60526316  9.78947368  9.97368421 10.15789474 10.34210526 10.52631579
 10.71052632 10.89473684 11.07894737 11.26315789 11.44736842 11.63157895
 11.81578947 12.        ]

Shen best fits (z = 4.0): AIC = 41.305378348889484 chi2 = 37.1816670086833
Best crit = 10.16,  Best pre-disk = 2.50, Best post-disk = 1.86,  Best slope = 1.30,  Best norm = 1.39,   Best local norm = -2.80
 


OSError: Unable to open file (unable to open file: name = 'output/chi2_SHEN_6param_eL_adjreso_z0.5.h5py', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [5]:
f = h5py.File("SHEN_obs_collect.h5py", "r")
for g in f:
    print('\n',g)
    for sg in f[g]:
        print('\t',sg)


 z=0.1
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=0.2
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=0.3
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=0.4
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=0.5
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=0.6
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=0.8
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=1.0
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=1.2
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=1.4
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=1.6
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=1.8
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=2.0
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=2.2
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=2.4
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=2.6
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=2.8
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=3.0
	 B-Band
	 Hard-X
	 Mid-IR
	 Soft-X
	 UV-1450A

 z=3.2
	 

In [43]:
def bestfit_QLF_6param(z, qlf_params, pdf):
    
    pfs = 8
    pars = 6
    cs = list(cm.Greens(np.arange(pars) / pars) ) 

    fig, (ax, ax4) = plt.subplots(1,2,figsize=(8,3.5))

    
    qlf = QLF(z, qlf_params['bins'])
    qlf.get_Mbh( qlf_params['Mscrit'], qlf_params['preslope'], qlf_params['Mbhnorm'], approx_local=True, norm_local = 11+qlf_params['postnorm'])
    qlf.get_dNdlnL(qlf_params['lums'], [qlf_params['presig'], qlf_params['postsig']])
    
    shendens, stanave, stanab, stanb, _ = Shen_fit_uncer(z, qlf_params['lums'])
    
    lumsp = 10**qlf_params['lums']*3.8e33
    
    
    bands = []
    labels = []
    colors = {'B-Band':'pink', 'UV-1450A':'seagreen', 'Hard-X':'royalblue', 'Soft-X':'gray', 'Mid-IR':'olive'}

    for band in fobs["z="+str(z)]:
        index = "z="+str(z)+"/"+band
        x = fobs[index]['x'][:]
        y = fobs[index]['y'][:]
        xerr = fobs[index]['xerr'][:]
        yerr = fobs[index]['yerr'][:]
        xerrcor = np.zeros((2,len(xerr)))
        xerrcor[1,:] = 10**(x + xerr) - 10**x
        xerrcor[0,:] = 10**x - 10**(x - xerr)

        b = ax.errorbar(10**x, y, yerr = yerr, xerr=xerrcor, c=colors[band], fmt = 'o', fillstyle='none', capsize=1, ms=1.5, lw=0.5)
        bands.append(b)
        labels.append(band)
    
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.5, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -15, np.log10(temp * np.log(10)), color = c, alpha = 0.2)
        mass_begin += 1
        
    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF') 
    ax.plot(lumsp, shendens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
#     ax.fill_between(lumsp, shendens-stanab-0.5, shendens+stanb+0.5, color='purple', alpha=.2)
    
    ax.axis([1e40,1e52,-12,-1])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**48.75,-5,'z = '+str(z),fontsize = pfs)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    legend = ax.legend(fontsize = 8, loc='upper right')
    
    ax.axvline(10**7.5*3.8e33,c='k',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')
    
#     minchi2 = qlf_params['minchi2']
#     AIC = qlf_params['AIC']
#     ax.text(10**48.75,-5, r'$\chi^2$ = '+f'{minchi2:.2f}', fontsize = pfs)
#     if AIC:
#         ax.text(10**48.75, -6, r'AIC = '+f'{AIC:.2f}', fontsize = pfs)

    
    ax4.plot(10**qlf.StellBins, 10**qlf.BHBins, c = 'k', label='Implemented')
    ax4.plot(10**qlf.StellBins, 10**(qlf.StellBins - 2.8), c = 'r', linestyle='dashed', label='Reference')
    for c, mass in zip(cs, [7,8,9,10,11,12,13]):
        ax4.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)
    
    
    bestcrit, bestpost, bestpre, bestslope, bestnorm, bestlocal = qlf_params['Mscrit'], qlf_params['postsig'], qlf_params['presig'], qlf_params['preslope'], qlf_params['Mbhnorm'], qlf_params['postnorm']
    ax4.text(10**7.2, 10**8.2, 'transition point = $10^{'+f'{bestcrit:.2f}'+'} M_\odot$', fontsize = pfs)
    ax4.text(10**7.2, 10**7.7, 'pre-disk $\sigma_{ln X} = $'+f'{bestpre:.2f}', fontsize = pfs)
    ax4.text(10**7.2, 10**7.2, 'post-disk $\sigma_{ln X} = $'+f'{bestpost:.2f}', fontsize = pfs)
    ax4.text(10**10.0, 10**5.2, r'pre-disk slope = '+f'{bestslope:.3f}', fontsize = pfs)
    ax4.text(10**8.5, 10**4.7, r'pre-disk normalization = $10^{'+f'{bestnorm:.2f}'+'} M_\odot$', fontsize = pfs)
    ax4.text(10**8.5, 10**4.2, r'post-disk normalization = '+f'{10**bestlocal:.4f}', fontsize = pfs)

    
    ax4.set_xlabel('$M_*(M_\odot)$',fontsize=12)
    ax4.set_ylabel('$M_{BH}(M_\odot)$',fontsize=12)
    ax4.set_yscale('log')
    ax4.set_xscale('log')
    ax4.set_xlim([10**7,10**12.5])
    ax4.set_ylim([10**3, 10**10])
    ax4.legend(fontsize = 8, framealpha=1)
    ax4.tick_params(axis='both', which='both', labelsize=10, direction='in')
    
    legendB = ax.legend(handles = bands, labels=labels, fontsize = 8, loc='lower left', framealpha=0.5, edgecolor = 'k')
    ax.add_artist(legend)

    
    AIC_tot = qlf_params['AIC_tot']
    chi2_tot = qlf_params['chi2_tot']
    plt.suptitle('2P Model Vaired Fit z='+str(z)+' (with Shen+2020 obs data).')
    plt.tight_layout(rect=[0, 0.03, 1, 0.9])
    
    pdf.savefig(fig)



def get_AIC(chi2, N, p):
    return chi2 + 2*p + 2*p*(p+1)/(N-p-1)

fobs = h5py.File("SHEN_obs_collect.h5py", "r")

plt.close('all')
%matplotlib widget
redshifts = ['0.5', '1.0', '2.0', '3.0', '4.0']

minval_tot = 0
for z in redshifts:
    FILE="output/chi2_SHEN_6param_eL_adjreso_z"+z+".h5py"
    f = h5py.File(FILE, "r") 
    chi2_grid = f['chi2_grid'][:].T
    f.close()
    
    chi2_grid[null] = 1e10
    minval_tot += np.amin(chi2_grid)
    
AIC_tot = get_AIC(minval_tot, 100*len(redshifts), 6*len(redshifts))


with PdfPages('plots/bestfit_plots/extendof_kinkpref/QLF_allowkink_freeall_wobs.pdf') as pdf:
    for z in redshifts:
        FILE="output/chi2_SHEN_6param_eL_adjreso_z"+z+".h5py"
        f = h5py.File(FILE, "r") 
        logMstar0 = f['logMstar0'][:]
        siglnX2 = f['siglnX2'][:]
        siglnX1 = f['siglnX1'][:]
        slope_low = f['slope_low'][:]
        norm_from_local = f['norm_from_local'][:]
        norm_of_local = f['norm_of_local'][:]
        chi2_grid = f['chi2_grid'][:].T
        f.close()
        
        chi2_grid[null] = 1e10

        minval = np.amin(chi2_grid)
        minind = np.where(chi2_grid == minval)
        AIC = get_AIC(minval, 100, 6)

        bestlocal = norm_of_local[minind[0][0]] 
        bestnorm = norm_from_local[minind[1][0]]
        bestslope = slope_low[minind[2][0]]
        bestpost = siglnX2[minind[3][0]]
        bestpre = siglnX1[minind[4][0]]
        bestcrit = logMstar0[minind[5][0]]
    
        print(f'\nShen best fits (z = {z}): AIC =',AIC, r'chi2 =',minval)
        print(f'Best crit = {bestcrit:.2f},  Best pre-disk = {bestpre:.2f}, Best post-disk = {bestpost:.2f},  Best slope = {bestslope:.2f},  Best norm = {bestnorm:.2f},   Best local norm = {bestlocal:.2f}\n ')

        qlf_params = {'bins':0.005, 'Mscrit':bestcrit, 'Mbhnorm':bestnorm, 'preslope':bestslope, 'presig':bestpre, 'postsig':bestpost, 'postnorm':bestlocal, 'lums':np.linspace(5,18,200),\
                      'minchi2':minval, 'chi2_tot':minval_tot, 'AIC':AIC, 'AIC_tot':AIC_tot}
        
        bestfit_QLF_6param(float(z), qlf_params, pdf)

        
fobs.close()



Shen best fits (z = 0.5): AIC = 13.912263995797511 chi2 = 1.0090381893459
Best crit = 10.71,  Best pre-disk = 2.29, Best post-disk = 1.43,  Best slope = 0.82,  Best norm = 1.75,   Best local norm = -2.30
 


FigureCanvasNbAgg()


Shen best fits (z = 1.0): AIC = 14.099931275727002 chi2 = 1.1967054692753898
Best crit = 10.71,  Best pre-disk = 1.86, Best post-disk = 1.86,  Best slope = 0.96,  Best norm = 1.93,   Best local norm = -2.35
 


FigureCanvasNbAgg()


Shen best fits (z = 2.0): AIC = 14.507869456312847 chi2 = 1.6046436498612344
Best crit = 10.71,  Best pre-disk = 1.64, Best post-disk = 1.64,  Best slope = 1.30,  Best norm = 2.11,   Best local norm = -2.40
 


FigureCanvasNbAgg()


Shen best fits (z = 3.0): AIC = 14.746204616213944 chi2 = 1.8429788097623319
Best crit = 10.71,  Best pre-disk = 1.86, Best post-disk = 1.43,  Best slope = 1.30,  Best norm = 1.93,   Best local norm = -2.35
 


FigureCanvasNbAgg()


Shen best fits (z = 4.0): AIC = 13.64597440491512 chi2 = 0.7427485984635087
Best crit = 10.34,  Best pre-disk = 2.29, Best post-disk = 1.64,  Best slope = 1.30,  Best norm = 1.57,   Best local norm = -2.60
 


FigureCanvasNbAgg()

In [50]:
def bestfit_QLF_2param(z, qlf_params, pdf):
    
    pfs = 8
    pars = 6
    cs = list(cm.Greens(np.arange(pars) / pars) ) 

    fig, (ax, ax4) = plt.subplots(1,2,figsize=(8,3.5))

    
    qlf = QLF(z, qlf_params['bins'])
    qlf.get_Mbh( qlf_params['Mscrit'], qlf_params['preslope'], qlf_params['Mbhnorm'], approx_local=True, norm_local = 11+qlf_params['postnorm'])
    qlf.get_dNdlnL(qlf_params['lums'], [qlf_params['presig'], qlf_params['postsig']])
    
    shendens, stanave, stanab, stanb, _ = Shen_fit_uncer(z, qlf_params['lums'])
    
    lumsp = 10**qlf_params['lums']*3.8e33
    
    
    bands = []
    labels = []
    colors = {'B-Band':'pink', 'UV-1450A':'seagreen', 'Hard-X':'royalblue', 'Soft-X':'gray', 'Mid-IR':'olive'}

    for band in fobs["z="+str(z)]:
        index = "z="+str(z)+"/"+band
        x = fobs[index]['x'][:]
        y = fobs[index]['y'][:]
        xerr = fobs[index]['xerr'][:]
        yerr = fobs[index]['yerr'][:]
        xerrcor = np.zeros((2,len(xerr)))
        xerrcor[1,:] = 10**(x + xerr) - 10**x
        xerrcor[0,:] = 10**x - 10**(x - xerr)

        b = ax.errorbar(10**x, y, yerr = yerr, xerr=xerrcor, c=colors[band], fmt = 'o', fillstyle='none', capsize=1, ms=1.5, lw=0.5)
        bands.append(b)
        labels.append(band)
    
    mass_begin = 7
    temp = np.zeros(len(lumsp))
    cs = list(cm.Greens(np.arange(pars) / pars) ) 
    for m, n, c in zip(qlf.StellBins, range(pars), cs):
        temp = temp*0
        for dens in np.transpose(qlf.intvals)[((qlf.StellBins >= mass_begin)&(qlf.StellBins<= mass_begin+1)),:]:
            temp += dens
        l = ax.plot(lumsp, np.log10(temp * np.log(10)), lw = 1.5, color = c, linestyle='dotted')
        ax.fill_between(lumsp, -15, np.log10(temp * np.log(10)), color = c, alpha = 0.2)
        mass_begin += 1
        
    ax.plot(lumsp, np.log10(qlf.dNdlogL), c='k', label = 'Predicted QLF') 
    ax.plot(lumsp, shendens, label='Shen Global A',c='purple',lw=2,alpha = 0.5)
#     ax.fill_between(lumsp, shendens-stanab-0.5, shendens+stanb+0.5, color='purple', alpha=.2)
    
    ax.axis([1e40,1e52,-12,-1])
    ax.set_xlabel(r'$L_{bol} (erg\ s^{-1})$', fontsize=12)
    ax.set_ylabel(r'$\log_{10} \Phi (Mpc^{-3} \log_{10} [L_{bol}]^{-1})$', fontsize =12)
    ax.text(10**48.75,-5,'z = '+str(z),fontsize = pfs)
    ax.set_xscale('log')
    ax.tick_params(axis='both', which='both', labelsize=10, direction='in')
    legend = ax.legend(fontsize = 8, loc='upper right')
    
    ax.axvline(10**7.5*3.8e33,c='k',linestyle='dotted')
    ax.axvline(10**14.95*3.8e33,c='k',linestyle='dotted')

    
    ax4.plot(10**qlf.StellBins, 10**qlf.BHBins, c = 'k', label='Implemented')
    ax4.plot(10**qlf.StellBins, 10**(qlf.StellBins - 2.8), c = 'r', linestyle='dashed', label='Observed')
    for c, mass in zip(cs, [7,8,9,10,11,12,13]):
        ax4.axvline(10**mass, ls = 'dotted', c=c)
        ax4.axvspan(10**mass, 10**(mass+1), color=c, alpha = 0.5)
    
    bestpost, bestlocal = qlf_params['postsig'], qlf_params['postnorm']
    ax4.text(10**7.2, 10**7.7, '$\sigma_{ln X} = $'+f'{bestpost:.2f}', fontsize = pfs)
    ax4.text(10**8, 10**4.2, r'post-disk normalization = '+f'{10**bestlocal:.4f}', fontsize = pfs)
    
    ax4.set_xlabel('$M_*(M_\odot)$',fontsize=12)
    ax4.set_ylabel('$M_{BH}(M_\odot)$',fontsize=12)
    ax4.set_yscale('log')
    ax4.set_xscale('log')
    ax4.set_xlim([10**7,10**12.5])
    ax4.set_ylim([10**3, 10**10])
    ax4.legend(fontsize = 8, framealpha=1)
    ax4.tick_params(axis='both', which='both', labelsize=10, direction='in')

    legendB = ax.legend(handles = bands, labels=labels, fontsize = 8, loc='lower left', framealpha=0.5, edgecolor = 'k')
    ax.add_artist(legend)
    
    AIC_tot = qlf_params['AIC_tot']
    chi2_tot = qlf_params['chi2_tot']
    plt.suptitle('Linear Model Vaired Fit z='+str(z)+' (with Shen+2020 obs data).')
    plt.tight_layout(rect=[0, 0.03, 1, 0.9])

#     pdf.savefig(fig)

def get_AIC(chi2, N, p):
    return chi2 + 2*p + 2*p*(p+1)/(N-p-1)

fobs = h5py.File("SHEN_obs_collect.h5py", "r")

plt.close('all')
%matplotlib widget
redshifts = ['0.5', '1.0', '2.0', '3.0', '4.0']


minval_tot = 0
for z in redshifts:
    FILE="output_072720/chi2_SHEN_2param_eL_medres_z"+z+".h5py"
    f = h5py.File(FILE, "r") 
    chi2_grid = f['chi2_grid'][:].T
    f.close()

    minval_tot += np.amin(chi2_grid)
AIC_tot = get_AIC(minval_tot, 100*len(redshifts), 2*len(redshifts))

with PdfPages('plots/bestfit_plots/extendof_kinkpref/QLF_forcelinear_freeall_wobs.pdf') as pdf:
    for z in redshifts:
        FILE="output_072720/chi2_SHEN_2param_eL_medres_z"+z+".h5py"
        f = h5py.File(FILE, "r") 
        logMstar0 = f['logMstar0'][:]
        siglnX2 = f['siglnX2'][:]
        siglnX1 = f['siglnX1'][:]
        slope_low = f['slope_low'][:]
        norm_from_local = f['norm_from_local'][:]
        norm_of_local = f['norm_of_local'][:]
        chi2_grid = f['chi2_grid'][:].T
        f.close()

        minval = np.amin(chi2_grid)
        minind = np.where(chi2_grid == minval)
        AIC = get_AIC(minval, 100, 2)

        bestlocal = norm_of_local[minind[0][0]] 
        bestnorm = norm_from_local[0]
        bestslope = slope_low[0]
        bestpost = siglnX2[minind[1][0]]
        bestpre = siglnX1[0]
        bestcrit = logMstar0[0]

        print(f'\nShen best fits (z = {z}): AIC =',AIC, r'chi2 =',minval)
        print(f'Best post-disk sigma = {bestpost:.2f}, Best local norm = {bestlocal:.2f}\n ')

        qlf_params = {'bins':0.005, 'Mscrit':bestcrit, 'Mbhnorm':bestnorm, 'preslope':bestslope, 'presig':bestpre, 'postsig':bestpost, 'postnorm':bestlocal, 'lums':np.linspace(5,18,200),\
                      'minchi2':minval, 'chi2_tot':minval_tot, 'AIC':AIC, 'AIC_tot':AIC_tot}
        
        bestfit_QLF_2param(float(z), qlf_params, pdf)
        
fobs.close()



Shen best fits (z = 0.5): AIC = 9.337988309503936 chi2 = 5.214276969297751
Best post-disk sigma = 2.42, Best local norm = -3.00
 


FigureCanvasNbAgg()


Shen best fits (z = 1.0): AIC = 14.938556129183699 chi2 = 10.814844788977513
Best post-disk sigma = 2.26, Best local norm = -3.00
 


FigureCanvasNbAgg()


Shen best fits (z = 2.0): AIC = 35.93362993748326 chi2 = 31.809918597277072
Best post-disk sigma = 2.11, Best local norm = -3.00
 


FigureCanvasNbAgg()


Shen best fits (z = 3.0): AIC = 46.72905560278501 chi2 = 42.605344262578825
Best post-disk sigma = 2.42, Best local norm = -3.00
 


FigureCanvasNbAgg()


Shen best fits (z = 4.0): AIC = 41.305378348889484 chi2 = 37.1816670086833
Best post-disk sigma = 2.42, Best local norm = -3.00
 


FigureCanvasNbAgg()

In [52]:
xtot = []
ytot = []
yerrtot = []
fobs = h5py.File("SHEN_obs_collect.h5py", "r")
for band in fobs["z="+str(z)]:
    index = "z="+str(z)+"/"+band
    xtot.extend(fobs[index]['x'][:]) ##log10 of bolometric L in erg/s
    ytot.extend(fobs[index]['y'][:]) ##log10 of QLF
    yerrtot.extend(fobs[index]['yerr'][:]) ##error on that
fobs.close()

print(xtot,'\n', ytot,'\n', yerrtot)


[47.7668016996971, 47.64810097967039, 47.52945498993454, 47.41086903632079, 47.29234890815285, 47.173900915965085, 47.05553193085257, 46.93724942523106, 46.866904376855786, 47.06400094610478, 47.261336456045456, 47.53794468265787, 45.35387571973461, 45.74152605475755, 46.13166200626239, 46.523702409623326, 45.488220658713956, 46.16213185259434, 46.84247497434496, 47.52708840799278, 44.19939760470395, 44.85179268546909, 45.516846617545546, 46.19108662125961, 46.871651307649564, 44.560464737925585, 44.888854110764115, 45.22035315059766, 45.55451183165563, 45.8909117110342, 46.229176220039236, 46.56897539943528, 46.910026350032865, 44.560464737925585, 44.888854110764115, 45.22035315059766, 45.55451183165563, 45.8909117110342, 46.229176220039236, 46.56897539943528, 46.910026350032865, 44.560464737925585, 45.38712707928198, 46.05983262448907, 44.91663345165338, 46.02137959623483, 46.699654835817945, 47.2457515785777, 47.864302932206726, 44.684856894500015, 45.20038384814087, 45.534395127402